## import libraries and functions load cleaned data

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

sys.path.append("..")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

from src.extraction import load_processed, save_processed, FEATURES, TARGET

df_clean = load_processed("data_cleand.csv")

df_clean

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.50,55.00,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.80,61.00,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.90,65.00,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.40,58.00,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.30,58.00,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...
49841,0.72,Ideal,D,SI1,60.80,57.00,2757,5.75,5.76,3.50
49842,0.72,Good,D,SI1,63.10,55.00,2757,5.69,5.75,3.61
49843,0.70,Very Good,D,SI1,62.80,60.00,2757,5.66,5.68,3.56
49844,0.86,Premium,H,SI2,61.00,58.00,2757,6.15,6.12,3.74


## variables

In [2]:
cat_col = ['cut', 'color', 'clarity']

cut_order = ["Fair","Good","Very Good","Premium","Ideal"]
color_order = ["J","I","H","G","F","E","D"]
clarity_order = ["I1","SI2","SI1","VS2","VS1","VVS2","VVS1","IF"]

num_col = ['carat', 'depth', 'table', 'x', 'y', 'z']

preprocessor = ColumnTransformer(
    transformers=[
        ("cat",OrdinalEncoder(categories=[cut_order, color_order, clarity_order]),cat_col),
        ("num", StandardScaler(),num_col)
    ]  
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

## Train/Test Split

In [3]:
X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test,y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X, y

(       carat        cut color clarity    x    y    z  depth  table
 0       0.23      Ideal     E     SI2 3.95 3.98 2.43  61.50  55.00
 1       0.21    Premium     E     SI1 3.89 3.84 2.31  59.80  61.00
 2       0.23       Good     E     VS1 4.05 4.07 2.31  56.90  65.00
 3       0.29    Premium     I     VS2 4.20 4.23 2.63  62.40  58.00
 4       0.31       Good     J     SI2 4.34 4.35 2.75  63.30  58.00
 ...      ...        ...   ...     ...  ...  ...  ...    ...    ...
 49841   0.72      Ideal     D     SI1 5.75 5.76 3.50  60.80  57.00
 49842   0.72       Good     D     SI1 5.69 5.75 3.61  63.10  55.00
 49843   0.70  Very Good     D     SI1 5.66 5.68 3.56  62.80  60.00
 49844   0.86    Premium     H     SI2 6.15 6.12 3.74  61.00  58.00
 49845   0.75      Ideal     D     SI2 5.83 5.87 3.64  62.20  55.00
 
 [49846 rows x 9 columns],
        price
 0        326
 1        326
 2        327
 3        334
 4        335
 ...      ...
 49841   2757
 49842   2757
 49843   2757
 49844   2757
 

## Model Training

In [4]:
models = {
    "Linear Regression" : LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
}

pipelines = {
    name : Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("regressor", model)
    ]
    )
    for name, model in models.items()
}


In [5]:
results = []

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    results.append({
        "Model" : name,
        "MAE"   : mean_absolute_error(y_test, y_pred),
        "RMSE"  : mean_squared_error(y_test, y_pred) ** 0.5,
        "R2"    : r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values(by="RMSE")

display(results_df)

c:\Users\lenovo pc\Desktop\diamond-price-ml\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,Model,MAE,RMSE,R2
1,Random Forest,266.78,541.50,0.98
0,Linear Regression,801.83,1214.73,0.91


## Cross-Validation GridSearchCV

In [6]:

param_grid_rf = {
    "regressor__n_estimators": [50, 100, 200],
    "regressor__max_depth": [None, 10, 20],
    "regressor__min_samples_split": [2, 5, 10]
}

param_grid_lr = {
    "regressor__fit_intercept": [True, False]
}

models_param_grid = {
    "Linear Regression": param_grid_lr,
    "Random Forest": param_grid_rf
}


for name, pipeline in pipelines.items():

    if name not in models_param_grid:
        continue
    print(f"\nTraining: {name}")

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=models_param_grid[name],
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    print(f"\nModel: {name}")
    print("Best parameters:", grid_search.best_params_)
    print("Best CV RMSE:", -grid_search.best_score_)



Training: Linear Regression

Model: Linear Regression
Best parameters: {'regressor__fit_intercept': True}
Best CV RMSE: 1204.5521378872497

Training: Random Forest


c:\Users\lenovo pc\Desktop\diamond-price-ml\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Model: Random Forest
Best parameters: {'regressor__max_depth': 20, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 200}
Best CV RMSE: 544.1847213718677
